# 14 — Semantic answer cache (phase 6)

When the same — or a very close — first-turn question comes back, serve the stored answer instead of running the agent again. Opt-in (`FACTORY_FLOOR_SEMANTIC_CACHE_ENABLED=true`).

Safety: keyed by machine / equipment / language / tenant; **fault-code questions need an exact match** (never a near-miss — 'F30021' must not be served for 'F30022'); only answers the safety gate lets through are cached; a `version_stamp` + TTL bound staleness.

This notebook uses a **temporary** cache dir and audit db.

In [1]:
import os, sys, tempfile
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

_tmp = Path(tempfile.mkdtemp())
os.environ['FACTORY_FLOOR_SEMANTIC_CACHE_ENABLED'] = 'true'
os.environ['FACTORY_FLOOR_SEMANTIC_CACHE_DIR'] = str(_tmp / 'qa_cache')
os.environ['FACTORY_FLOOR_AUDIT_DB_PATH'] = str(_tmp / 'audit.sqlite3')

from factory_floor.config import COLLECTION_NAME, VECTOR_DIR, get_settings
get_settings.cache_clear()
from factory_floor import services
from factory_floor.cache import SemanticCache
from factory_floor.vectorstore import get_embeddings, load_vectorstore

vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=get_embeddings())
print('cache enabled:', get_settings().semantic_cache_enabled, '| threshold:', get_settings().semantic_cache_similarity_threshold)

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cache enabled: True | threshold: 0.95


## First ask — a miss, runs the agent

In [2]:
import time
req = services.DiagnosticRequest(
    question_text='The motor is overheating and vibrating during normal load. What checks are supported by the manuals?',
    machine_id='MOTOR-01', equipment_type='electric_motor', operator_id='OP-1001', language='English',
)
t0 = time.time(); first = services.run_diagnostic(req, vectorstore=vectorstore); t_first = time.time() - t0
print(f'cache_hit={first.cache_hit}  cost=${first.cost["total_usd"]:.5f}  n_calls={first.cost["n_calls"]}  {t_first:.1f}s')
print(first.answer[:400])

cache_hit=False  cost=$0.00282  n_calls=3  11.3s
Safety precautions:
- Before performing any physical inspection or measurement, isolate and de-energize the motor.
- Apply lockout/tagout procedures to ensure the motor cannot be accidentally energized.
- Wait for the DC link capacitors to discharge before touching any terminals.
- Verify absence of voltage with appropriate testing equipment.
- Only qualified personnel should perform these checks.


In [3]:
assert first.cache_hit is False and first.cost['n_calls'] >= 1

## Ask a paraphrase — a hit, ~0 cost, same citations

In [4]:
para = services.DiagnosticRequest(
    question_text='the motor runs hot and shakes under load - what does the documentation say to check?',
    machine_id='MOTOR-01', equipment_type='electric_motor', operator_id='OP-1001', language='English',
)
# NOTE: whether a paraphrase hits depends on the 0.95 threshold and the embedding of
# the two phrasings. An exact repeat always hits; a loose paraphrase may not.
t0 = time.time(); para_res = services.run_diagnostic(para, vectorstore=vectorstore); t_para = time.time() - t0
print(f'paraphrase: cache_hit={para_res.cache_hit}  {t_para:.2f}s')

t0 = time.time(); exact = services.run_diagnostic(req, vectorstore=vectorstore); t_exact = time.time() - t0
print(f'exact repeat: cache_hit={exact.cache_hit}  cost=${exact.cost["total_usd"]:.5f}  n_calls={exact.cost["n_calls"]}  {t_exact:.2f}s')
assert exact.cache_hit is True
assert exact.cost['n_calls'] == 0
assert exact.answer == first.answer and exact.sources == first.sources

paraphrase: cache_hit=False  12.01s


exact repeat: cache_hit=True  cost=$0.00000  n_calls=0  0.22s


## Fault-code safety — a near-miss code is not served

In [5]:
code_q = services.DiagnosticRequest(question_text='F30021 ground fault, what should be checked?',
    machine_id='GENERAL', equipment_type='VFD', operator_id='OP-1001', language='English')
services.run_diagnostic(code_q, vectorstore=vectorstore)
same = services.run_diagnostic(code_q, vectorstore=vectorstore)
assert same.cache_hit is True, 'exact code repeat should hit'

other = services.DiagnosticRequest(question_text='F30022 ground fault, what should be checked?',
    machine_id='GENERAL', equipment_type='VFD', operator_id='OP-1001', language='English')
other_res = services.run_diagnostic(other, vectorstore=vectorstore)
assert other_res.cache_hit is False, 'F30022 must NOT be served from the F30021 entry'
print('fault-code exact-match rule holds — OK')

fault-code exact-match rule holds — OK


## Threshold sweep (why 0.95)

In [6]:
cache = SemanticCache(embeddings=get_embeddings())
pairs = [
    ('the motor is overheating and vibrating', 'the motor runs hot and shakes'),
    ('the motor is overheating and vibrating', 'how do I reset a tripped VFD'),
    ('the motor is overheating and vibrating', 'the motor is overheating and vibrating badly'),
]
for a, b in pairs:
    cache._store.add_texts([a], metadatas=[{'x':'1'}], ids=[a])
    (_doc, dist), = cache._store.similarity_search_with_score(b, k=1)
    print(f'sim={1-dist:.3f}   {a!r:55} vs {b!r}')
print('\n0.95 keeps close paraphrases, rejects unrelated questions and loose ones.')

sim=0.824   'the motor is overheating and vibrating'                vs 'the motor runs hot and shakes'


sim=0.445   'the motor is overheating and vibrating'                vs 'how do I reset a tripped VFD'


sim=0.958   'the motor is overheating and vibrating'                vs 'the motor is overheating and vibrating badly'

0.95 keeps close paraphrases, rejects unrelated questions and loose ones.


## Clearing the cache

In [7]:
c = SemanticCache(embeddings=get_embeddings())
print('entries before clear:', c.count())
c.clear()
print('entries after clear :', c.count())
assert c.count() == 0

entries before clear: 5
entries after clear : 0
